# 02 · Fine-tune BERT and evaluate it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baz08/crypto-sentiment-bert/blob/main/notebooks/02_train_evaluate.ipynb)

Fine-tunes `bert-base-uncased` into a 3-class sentiment classifier (negative / neutral / positive) for crypto
Reddit text, and checks whether it beats two simpler baselines:

1. **Majority class**: always predict the most common label.
2. **TF-IDF + logistic regression**: a classic bag-of-words model.

**Before running:** *Runtime → Change runtime type → T4 GPU*. Training takes a few minutes on a T4. On a CPU it
works but takes much longer.

The code matches `deployment/bert_training/berttest.py`; this notebook adds plots and error analysis.

In [ ]:
# transformers v5 dropped TensorFlow and changed tokenizers; stay on 4.x so the
# saved model loads in the API (which pins transformers 4.46.3).
!pip install -q "transformers>=4.46,<5"

In [ ]:
# Get the repo (for the labeled data). Change BRANCH to test an unmerged branch.
REPO_URL = "https://github.com/baz08/crypto-sentiment-bert"
BRANCH = "main"

import os
if not os.path.exists("crypto-sentiment-bert"):
    !git clone -q --depth 1 -b {BRANCH} {REPO_URL}
REPO_DIR = "crypto-sentiment-bert"
DATA_DIR = f"{REPO_DIR}/deployment/reddit"

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from sklearn import metrics
from sklearn.model_selection import train_test_split

CONFIG = {
    "base_model": "bert-base-uncased",
    "max_length": 256,   # tokens; same limit the API uses
    "epochs": 2,
    "batch_size": 16,
    "lr": 2e-5,
    "test_size": 0.2,
    "seed": 42,
}
LABELS = ["Negative", "Neutral", "Positive"]

transformers.set_seed(CONFIG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__}, transformers {transformers.__version__}, device: {device}")
if device == "cpu":
    print("No GPU found: Runtime -> Change runtime type -> T4 GPU (training on CPU will be slow).")

## 1. Load data and split

A **shuffled, label-stratified** 80/20 split: the test set has the same label mix as the full data. (The original
notebook held out the last 500 rows, which all came from one source and were ~75% neutral.)

In [ ]:
data = pd.read_csv(f"{DATA_DIR}/Crypto_c.csv")

train_df, test_df = train_test_split(
    data, test_size=CONFIG["test_size"], random_state=CONFIG["seed"], shuffle=True, stratify=data["LABEL_COLUMN"]
)
train_df, test_df = train_df.reset_index(drop=True), test_df.reset_index(drop=True)

pd.DataFrame({
    "train": train_df["LABEL_COLUMN"].map(dict(enumerate(LABELS))).value_counts(),
    "test": test_df["LABEL_COLUMN"].map(dict(enumerate(LABELS))).value_counts(),
}).loc[LABELS]

## 2. Baselines

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

def compute_metrics(y_true, y_pred):
    y_true, y_pred = list(y_true), list(y_pred)
    return {
        "accuracy": metrics.accuracy_score(y_true, y_pred),
        "macro_f1": metrics.f1_score(y_true, y_pred, labels=[0, 1, 2], average="macro", zero_division=0),
        "per_class_f1": dict(zip(LABELS, metrics.f1_score(y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0).tolist())),
        "confusion_matrix": metrics.confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist(),
    }

y_train, y_test = train_df["LABEL_COLUMN"], test_df["LABEL_COLUMN"]
results = {}

# 1) majority class
majority = y_train.mode()[0]
results["majority_baseline"] = compute_metrics(y_test, [majority] * len(y_test))

# 2) TF-IDF + logistic regression
tfidf = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=CONFIG["seed"]),
)
tfidf.fit(train_df["DATA_COLUMN"], y_train)
tfidf_pred = tfidf.predict(test_df["DATA_COLUMN"])
results["tfidf_logreg"] = compute_metrics(y_test, tfidf_pred)

for name, r in results.items():
    print(f"{name:18s} accuracy {r['accuracy']:.3f}   macro-F1 {r['macro_f1']:.3f}")

## 3. Fine-tune BERT

Standard fine-tuning recipe: AdamW with a small learning rate (2e-5), linear warmup and decay, and gradient
clipping, so the pretrained weights aren't wrecked in the first few steps.

In [ ]:
from torch.utils.data import DataLoader
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)

tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["base_model"],
    num_labels=len(LABELS),
    id2label=dict(enumerate(LABELS)),
    label2id={label: i for i, label in enumerate(LABELS)},
).to(device)

def make_loader(texts, labels, shuffle):
    enc = tokenizer(list(texts), truncation=True, max_length=CONFIG["max_length"])
    items = [{**{k: v[i] for k, v in enc.items()}, "labels": int(y)} for i, y in enumerate(labels)]
    return DataLoader(items, batch_size=CONFIG["batch_size"], shuffle=shuffle,
                      collate_fn=DataCollatorWithPadding(tokenizer),
                      generator=torch.Generator().manual_seed(CONFIG["seed"]))

def predict(texts, batch_size=64):
    model.eval()
    texts, preds = list(texts), []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            enc = tokenizer(texts[start:start + batch_size], truncation=True, max_length=CONFIG["max_length"],
                            padding=True, return_tensors="pt").to(device)
            preds.extend(model(**enc).logits.argmax(dim=-1).cpu().tolist())
    return preds

train_loader = make_loader(train_df["DATA_COLUMN"], y_train, shuffle=True)

In [ ]:
total_steps = len(train_loader) * CONFIG["epochs"]
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], eps=1e-8)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

history = []
for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        running_loss += loss.item()

    val_pred = predict(test_df["DATA_COLUMN"])
    history.append({
        "epoch": epoch,
        "train_loss": running_loss / len(train_loader),
        "val_accuracy": metrics.accuracy_score(y_test, val_pred),
        "val_macro_f1": metrics.f1_score(y_test, val_pred, average="macro"),
    })
    print(history[-1])

## 4. Results

In [ ]:
bert_pred = predict(test_df["DATA_COLUMN"])
results["bert"] = compute_metrics(y_test, bert_pred)

summary = pd.DataFrame({
    name: {"accuracy": r["accuracy"], "macro-F1": r["macro_f1"], **{f"F1 {k}": v for k, v in r["per_class_f1"].items()}}
    for name, r in results.items()
}).T.round(3)
summary

In [ ]:
print(metrics.classification_report(y_test, bert_pred, target_names=LABELS, digits=3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (name, pred) in zip(axes, [("TF-IDF + LogReg", tfidf_pred), ("BERT", bert_pred)]):
    metrics.ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=LABELS, normalize="true", values_format=".2f",
        cmap="Blues", colorbar=False, ax=ax,
    )
    ax.set_title(f"{name} (rows sum to 1)")
plt.tight_layout()
plt.show()

### Accuracy by source

The four sources were labeled differently (`sentiment.csv` has no neutral class), so check whether the model does
much better on some sources than others.

In [ ]:
by_source = (
    test_df.assign(correct=np.array(bert_pred) == y_test.values)
    .groupby("source")
    .agg(rows=("correct", "size"), bert_accuracy=("correct", "mean"))
    .round(3)
)
by_source

### Error analysis

Read a sample of mistakes: sarcasm, mixed opinions and label noise usually show up here, and they make good
material for the README's limitations section.

In [ ]:
errors = test_df.assign(predicted=[LABELS[p] for p in bert_pred], actual=y_test.map(dict(enumerate(LABELS))))
errors = errors[errors["predicted"] != errors["actual"]]
print(f"{len(errors)} errors out of {len(test_df)}")

pd.set_option("display.max_colwidth", 200)
errors.sample(min(12, len(errors)), random_state=CONFIG["seed"])[["source", "actual", "predicted", "DATA_COLUMN"]]

## 5. Save metrics and the model

In [ ]:
results["config"] = {**CONFIG, "train_rows": len(train_df), "test_rows": len(test_df),
                     "transformers": transformers.__version__, "torch": torch.__version__}
results["history"] = history
with open("metrics.json", "w") as f:
    json.dump(results, f, indent=2)

model.save_pretrained("model")
tokenizer.save_pretrained("model")
print("saved metrics.json and model/")

try:
    from google.colab import files
    files.download("metrics.json")
except ImportError:
    pass

## 6. (Optional) Publish to the Hugging Face Hub

The API (`deployment/api`) downloads the model from the Hub. Running this cell asks for a Hugging Face access
token with **write** permission (huggingface.co → Settings → Access Tokens).

In [ ]:
HUB_REPO = "baz08/crypto-sentiment-bert"   # the API's default MODEL_REPO
PUSH = False                                # set to True to upload

if PUSH:
    from huggingface_hub import notebook_login
    notebook_login()
    model.push_to_hub(HUB_REPO)
    tokenizer.push_to_hub(HUB_REPO)
    print(f"https://huggingface.co/{HUB_REPO}")

## 7. Try it

Runs new text through the same cleaning the API uses, then the fine-tuned model.

In [ ]:
"""Light text cleaning applied to every comment before it reaches BERT.

This is the cleaning used for BOTH training (``merge_clean.py``) and
inference (``deployment/api/ML/textprep.py`` is an identical copy, kept in
sync by ``tests/test_textprep.py``). Using the same function in both places
avoids train/serve skew: the model sees the same kind of text in production
that it was trained on.

It is deliberately much lighter than ``cleaner.normalize_corpus``. BERT was
pretrained on natural text, so stopwords, punctuation, and casing all carry
signal ("not good" vs "good", "!!!"), and ``bert-base-uncased`` lowercases
on its own. We only remove things that are pure noise: URLs, HTML entities
left over from Reddit's markdown, encoding artifacts, and extra whitespace.

Standard library only, so it can be imported by the API container and the
data pipeline without extra dependencies.
"""

import html
import re

URL_PATTERN = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
MARKDOWN_HEADER_PATTERN = re.compile(r"^#{1,6}\s*", flags=re.MULTILINE)
WHITESPACE_PATTERN = re.compile(r"\s+")

# Characters that show up when UTF-8 text was decoded as cp1252/latin-1
# (e.g. "isnâ€™t" instead of "isn't").
MOJIBAKE_MARKERS = ("â€", "Ã", "Â")


def fix_mojibake(text):
    """Best-effort repair of UTF-8 text that was mis-decoded as cp1252."""
    if not any(marker in text for marker in MOJIBAKE_MARKERS):
        return text
    try:
        # cp1252 leaves five byte values undefined (0x81, 0x8D, 0x8F, 0x90,
        # 0x9D); those survive mis-decoding as raw control characters, so
        # fall back to latin-1 for them byte-by-byte.
        raw = b"".join(_to_byte(ch) for ch in text)
        return raw.decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text  # not actually mojibake (or mixed); leave it alone


def _to_byte(ch):
    try:
        return ch.encode("cp1252")
    except UnicodeEncodeError:
        return ch.encode("latin-1")


def clean_for_model(text):
    """Normalize a single comment for BERT. Returns '' for missing input."""
    if not isinstance(text, str):
        return ""
    text = fix_mojibake(text)
    # Reddit exports are sometimes double-escaped ("&amp;#x200B;"), so unescape twice.
    text = html.unescape(html.unescape(text))
    text = text.replace("​", " ")
    text = URL_PATTERN.sub(" ", text)
    text = MARKDOWN_HEADER_PATTERN.sub("", text)
    text = WHITESPACE_PATTERN.sub(" ", text)
    return text.strip()

In [ ]:
samples = [
    "I love bitcoin, this dip is a gift",
    "ETH gas fees are absolutely killing me",
    "What wallet do you all use for cold storage?",
    "Great, another rug pull. Exactly what I needed today.",
]
probs = []
model.eval()
with torch.no_grad():
    for text in samples:
        enc = tokenizer(clean_for_model(text), truncation=True, max_length=CONFIG["max_length"], return_tensors="pt").to(device)
        probs.append(torch.softmax(model(**enc).logits, dim=-1)[0].cpu().numpy())

pd.DataFrame(probs, columns=LABELS, index=samples).round(3).assign(prediction=lambda d: d[LABELS].idxmax(axis=1))